In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"

import pathlib
from functools import partial

import time
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
# import matplotlib as mpl
# mpl.rcParams['text.usetex'] = True
# mpl.rcParams['text.latex.preamble']=r"\usepackage{bm}"

In [ ]:
import jax
import jax.numpy as jnp
# jax.config.update("jax_enable_x64", True)
gpus = jax.devices()
jax.config.update("jax_default_device", gpus[0])

import diffrax
import equinox as eqx
import optax

from haiku import PRNGSequence

In [ ]:
import exciting_environments as excenvs

import dmpe
from dmpe.models.models import NeuralEulerODEPendulum, NeuralODEPendulum, NeuralEulerODE, NeuralEulerODECartpole
from dmpe.models.model_utils import simulate_ahead_with_env
from dmpe.models.model_training import ModelTrainer
from dmpe.excitation.excitation_utils import loss_function, Exciter
from dmpe.algorithms.algorithm_utils import interact_and_observe

from dmpe.utils.density_estimation import (
    update_density_estimate_single_observation, update_density_estimate_multiple_observations, DensityEstimate
)
from dmpe.utils.signals import aprbs
from dmpe.evaluation.plotting_utils import (
    plot_sequence, append_predictions_to_sequence_plot, plot_sequence_and_prediction, plot_model_performance,
    plot_feature_combinations,
)
from dmpe.evaluation.experiment_utils import (
    get_experiment_ids, load_experiment_results, quick_eval, evaluate_experiment_metrics, evaluate_algorithm_metrics, load_all_experiment_results
)
from dmpe.utils.density_estimation import select_bandwidth
from dmpe.evaluation.experiment_utils import default_jsd, default_ae, default_mcudsa, default_ksfc

In [ ]:
from dmpe.data_management import DataPaths
from dmpe.utils.env_utils.fluid_tank_utils import setup_env as setup_fluid_tank_env
from dmpe.utils.env_utils.pendulum_utils import setup_env as setup_pendulum_env
from dmpe.utils.env_utils.cart_pole_utils import setup_env as setup_cart_pole_env

In [ ]:
env, penalty_function, _ = setup_cart_pole_env()

In [ ]:
seed = 0
n_time_steps = 15_000
n_tries = 4_000

# ---- # 

obs, state = env.reset(env.env_properties)
dim_obs_space = obs.shape[0]
dim_action_space = env.action_dim

observations = jnp.zeros((n_time_steps, dim_obs_space))
observations = observations.at[0].set(obs)
actions = jnp.zeros((n_time_steps - 1, dim_action_space))


key = jax.random.key(seed)
key, action_key = jax.random.split(key)

action = jax.random.normal(action_key, shape=(env.action_dim,))

In [ ]:
@partial(jax.jit, static_argnums=(0, 1))
def choose_action(env, penalty_function, proposed_actions, state, choice_key):
    """Choose randomly among the proposed actions that keep the system within bounds for the next step.
    If none of the inputs keep the systems in bounds, apply the one that causes the least penalty.
    
    This is a heursitic implmentation that uses an oracle to ensure compliance with the bounds, but chooses
    mostly randomly among the actions.
    """
    test_obs, test_state = jax.vmap(env.step, in_axes=(None, 0, None))(state, proposed_actions, env.env_properties)
    penalty_values = jax.vmap(penalty_function, in_axes=(0, 0))(test_obs[:, None, :], proposed_actions[:, None, :])

    def true_fun(key, data_array, penalty_values):
        """There are not options that keep the system within bounds. Apply the one with the least penalty."""
        idx_min_penalty = jnp.argmin(penalty_values)
        return data_array[idx_min_penalty]

    def false_fun(key, data_array, penalty_values):
        """There are actions that keep the system within bounds. Choose one randomly."""
        valid_points_bool = penalty_values == 0
        prob_points = valid_points_bool.astype(jnp.float32) / jnp.sum(valid_points_bool)
        return jax.random.choice(choice_key, proposed_actions, p=prob_points, axis=0)
       
    return jax.lax.cond(jnp.all(penalty_values != 0), true_fun, false_fun, *(choice_key, proposed_actions, penalty_values))

In [ ]:
for k in tqdm(range(n_time_steps)):

    key, action_key, choice_key = jax.random.split(key, 3)
    proposed_actions = action + jax.random.normal(action_key, shape=(n_tries, env.action_dim,))

    action = choose_action(env, penalty_function, proposed_actions, state, choice_key)
    
    next_obs, next_state, actions, observations = interact_and_observe(
        env=env, k=jnp.array([k]), action=action, state=state, actions=actions, observations=observations
    )

    state = next_state
    obs = next_obs

    if k % 5000 == 0 and k > 0:
        fig = plot_sequence(observations[:k], actions[:k], env.tau, env.obs_description, env.action_description)
        plt.show()

In [ ]:
plot_sequence(observations, actions, env.tau, env.obs_description, env.action_description)

In [ ]:
plot_feature_combinations(
    data = jnp.concatenate([observations[:-1], actions], axis=-1),
    labels = ["d", "v", "theta", "omega", "u"]
)
plt.show()